In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

In [5]:
import sdnq
import gc, os, time, urllib.request, zlib, torch, gradio as gr
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageEnhance, ImageFilter
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from diffusers import ModularPipeline

QWEN_ID = "Qwen/Qwen3-VL-4B-Instruct"
KLEIN_ID = "OzzyGT/Flux_3_Image_Klein_sdnq_dynamic_4bit"
REF_SIZE = 512  # references size

SYSTEM = """You write captions for a FLUX 3 Imag Klein generator.
The FIRST image is the PRODUCT: it must appear clearly, with its exact shape, color, material and hardware.
The remaining images, if exist, are a MOOD BOARD: describe all you can see, borrowing their setting, lighting, palette, camera feel, composition, mood and attitude.
But EXPLICIT CONTROLS from the user ALWAYS override the mood board.
Write ONE full scene caption in this order: medium and style, scene, lighting and camera, each subject with colors, materials and position in the frame.
Make it a high-fashion editorial realistic campaign, not a catalog photo. No bullet points, no preamble. Output only the caption."""

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def load_images(product_path, mood_paths, size):
    paths = [product_path] + list(mood_paths or [])
    ims = [Image.open(p).convert("RGB") for p in paths]
    for im in ims:
        im.thumbnail((size, size))
    return ims

FRAMING = {
    "full-body": "Mandatory: full-length vertical shot, where the entire body is visible from head to toe including the shoes, the model fills about two thirds of the frame height, with clear space above her head and below her feet.",
    "three-quarter length": "Mandatory: three-quarter length shot, where the frame cuts mid-thigh, the head, torso and hips are fully visible, with space above the head.",
    "waist-up": "Mandatory: waist-up shot, where the frame cuts at the waist, with the head, shoulders, arms and torso fully visible.",
    "close-up detail": "Mandatory: close-up detail shot, where the frame is tightly cropped on the product, which fills most of the image, with its texture, stitching, material and hardware in sharp focus; only a small part of the model is visible and the full body is not shown.",
    "close-up face": "Mandatory: close-up portrait, where the face fills most of the frame, cropped at the shoulders, eyes in sharp focus.",
}

# Step 1: Qwen writes the caption proposition
def write_caption(product_path, mood_paths, request, framing, lighting, camera, setting, palette):
    if not product_path:
        raise gr.Error("Upload a product image first.")
    images = load_images(product_path, mood_paths, 1024)
    
    framing = FRAMING.get(framing, framing)
    controls = {"framing": framing, "lighting": lighting, "camera": camera,
                "setting": setting, "palette": palette}
    controls_text = "\n".join(f"- {k}: {v}" for k, v in controls.items() if v and v.strip()) or "None."
    text = (f"Reference images follow in order: first is the PRODUCT; remaining images are MOOD BOARD inspiration.\n"
            f"User request: {request}\n"
            f"Use the mood board images only for broad visual direction.\n"
            f"Explicit controls (must override the mood board characteristics):\n{controls_text}\n"
            f"Output the caption.")

    model = Qwen3VLForConditionalGeneration.from_pretrained(QWEN_ID, dtype=torch.float16, device_map="auto")
    processor = AutoProcessor.from_pretrained(QWEN_ID)
    try:
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": [{"type": "image", "image": im} for im in images]
                                        + [{"type": "text", "text": text}]},
        ]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=450, do_sample=True, temperature=0.7)
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
        caption = processor.batch_decode(trimmed, skip_special_tokens=True,
                                         clean_up_tokenization_spaces=False)[0].strip()
    finally:
        del model, processor
        free_gpu()
    return caption

SDNQ: OpenVINO MM kernels are not available! Falling back to PyTorch Eager kernels for CPU device. Error message: No module named 'openvino'


In [7]:
# (width, height)
SIZES = {
    "2:3": (512, 768),
    "3:4": (576, 768),
    "4:5": (640, 800),
    "1:1": (640, 640),
    "4:3": (768, 576),
    "16:9": (768, 448),
}

# Step 2: Klein generates an image
def generate(product_path, mood_paths, caption, use_mood_ref, seed, aspect):
    width, height = SIZES[aspect]
    if not product_path:
        raise gr.Error("Upload a product image first.")
    if not caption or not caption.strip():
        raise gr.Error("The caption is empty. Write or generate a caption first.")
    
    mood_refs = list(mood_paths or [])[:3] if use_mood_ref else []
    refs = load_images(product_path, mood_refs, REF_SIZE)

    blocks = ModularPipeline.from_pretrained(KLEIN_ID).blocks
    blocks.sub_blocks.pop("prompt_upsampler")
    pipe = blocks.init_pipeline(KLEIN_ID)
    pipe.load_components(dtype=torch.bfloat16)
    pipe.to("cuda")

    try:
        free_gpu()
        img = pipe(prompt=caption, image=refs, height=int(height), width=int(width),
                   generator=torch.Generator().manual_seed(int(seed)), output="images")[0]
        img.save(f"/kaggle/working/campaign_seed{int(seed)}.png")
    except Exception as e:
        if "out of memory" in str(e).lower():
            raise gr.Error("Out of GPU memory.")
        raise
    finally:
        del pipe, blocks
        free_gpu()
    return img

In [15]:
# EDIT_SYSTEM = """You revise a complete scene caption for one localized image edit.
# The input image has a semi-transparent red overlay showing the exact area the user selected. The red overlay is only an editing guide; it is not part of the image and must not appear in the caption.
# Apply the user's edit instruction to the object or feature under the red selection. State the requested new appearance explicitly and faithfully. Never omit the requested change or describe the old appearance for that feature.
# Preserve everything outside the selected area: the model's identity and face, pose, framing, clothing, product, setting, lighting, colors, and editorial style. Keep the existing composition and camera unchanged. If the instruction conflicts with what is visible, follow the instruction only for the selected feature and trust the image everywhere else.
# Return one complete scene caption, not just an edit command. No bullets or preamble. Output only the caption."""

# # Step 3a: Create a selected-area edit prompt

# def edit_caption(caption, instruction, img, editor_value):
#     if img is None:
#         raise gr.Error("Generate an image first.")
#     if not instruction or not instruction.strip():
#         raise gr.Error("Describe the change first.")
#     mask = mask_from_editor(editor_value, img.size)
#     if mask is None:
#         raise gr.Error("Paint over the area you want to change first.")

#     pic = img.convert("RGB").copy()
#     pic.thumbnail((1024, 1024))
#     selected = mask.resize(pic.size, Image.Resampling.BILINEAR)
#     red_overlay = Image.new("RGBA", pic.size, (255, 0, 0, 0))
#     red_overlay.putalpha(selected.point(lambda value: int(value * 0.45)))
#     selected_pic = Image.alpha_composite(pic.convert("RGBA"), red_overlay).convert("RGB")

#     model = Qwen3VLForConditionalGeneration.from_pretrained(
#         QWEN_ID, dtype=torch.float16, device_map="auto"
#     )
#     processor = AutoProcessor.from_pretrained(QWEN_ID)
#     try:
#         messages = [
#             {"role": "system", "content": [{"type": "text", "text": EDIT_SYSTEM}]},
#             {"role": "user", "content": [
#                 {"type": "image", "image": selected_pic},
#                 {"type": "text", "text": f"Existing caption:\n{caption}\n\nEdit instruction for the red-selected area only: {instruction}\n\nWrite the full updated scene caption."}
#             ]},
#         ]
#         inputs = processor.apply_chat_template(
#             messages,
#             tokenize=True,
#             add_generation_prompt=True,
#             return_dict=True,
#             return_tensors="pt"
#         ).to(model.device)
#         out = model.generate(**inputs, max_new_tokens=500, do_sample=False)
#         trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
#         new_caption = processor.batch_decode(
#             trimmed,
#             skip_special_tokens=True,
#             clean_up_tokenization_spaces=False
#         )[0].strip()
#     finally:
#         del model, processor
#         free_gpu()
#     return new_caption


def generate_and_store(*args):
    img = generate(*args)
    return img, img


# def image_to_editor_value(img):
#     if img is None:
#         return None
#     background = img.convert("RGB")
#     blank_layer = Image.new("RGBA", background.size, (0, 0, 0, 0))
#     return {"background": background, "layers": [blank_layer], "composite": background}


# def reveal_edit_panel(img):
#     if img is None:
#         raise gr.Error("Generate an image first.")
#     return gr.update(visible=True), image_to_editor_value(img)


# def mask_from_editor(editor_value, size):
#     if not editor_value or not isinstance(editor_value, dict):
#         return None

#     mask = None
#     for layer in editor_value.get("layers") or []:
#         if layer is None:
#             continue
#         if isinstance(layer, np.ndarray):
#             layer = Image.fromarray(layer)

#         alpha = np.asarray(layer.convert("RGBA"))[:, :, 3]
#         mask = alpha if mask is None else np.maximum(mask, alpha)

#     if mask is None or mask.max() == 0:
#         return None

#     mask = (
#         mask.astype(np.float32) * (255.0 / float(mask.max()))
#     ).clip(0, 255).round().astype(np.uint8)

#     return Image.fromarray(mask, mode="L").resize(
#         size, Image.Resampling.BILINEAR
#     )


# def edit_and_store(img, editor_value, caption, add_prod, product_path, seed, feather):
#     new_img = edit_image(
#         img, editor_value, caption, add_prod, product_path, seed, feather
#     )
#     return new_img, new_img


# # Step 3b: Generate a replacement, then composite only inside the painted mask.
# def edit_image(base_img, editor_value, new_caption, add_product, product_path, seed, feather):
#     if base_img is None:
#         raise gr.Error("Generate an image first.")
#     if not new_caption or not new_caption.strip():
#         raise gr.Error("The updated prompt is empty.")

#     mask = mask_from_editor(editor_value, base_img.size)
#     if mask is None:
#         raise gr.Error("Paint over the area you want to change first.")

#     width, height = base_img.size
#     gen_width, gen_height = (width // 16) * 16, (height // 16) * 16
#     if gen_width < 16 or gen_height < 16:
#         raise gr.Error("The image is too small to edit.")

#     base = base_img.convert("RGB").copy()
#     base.thumbnail((768, 768))
#     refs = [base]
#     if add_product and product_path:
#         refs += load_images(product_path, [], REF_SIZE)

#     blocks = ModularPipeline.from_pretrained(KLEIN_ID).blocks
#     blocks.sub_blocks.pop("prompt_upsampler")
#     pipe = blocks.init_pipeline(KLEIN_ID)
#     pipe.load_components(dtype=torch.bfloat16)
#     pipe.to("cuda")

#     try:
#         free_gpu()
#         edited = pipe(
#             prompt=new_caption,
#             image=refs,
#             height=gen_height,
#             width=gen_width,
#             generator=torch.Generator().manual_seed(int(seed)),
#             output="images"
#         )[0]
#     except Exception as e:
#         if "out of memory" in str(e).lower():
#             raise gr.Error(
#                 "Out of GPU memory. Turn off the product reference, then restart the session."
#             )
#         raise
#     finally:
#         del pipe, blocks
#         free_gpu()

#     edited = edited.convert("RGB").resize(
#         base_img.size, Image.Resampling.LANCZOS
#     )
#     feather = float(feather)
#     soft_mask = (
#         mask.filter(ImageFilter.GaussianBlur(feather))
#         if feather > 0 else mask
#     )
#     result = Image.composite(edited, base_img.convert("RGB"), soft_mask)
#     result.save(f"/kaggle/working/edit_{int(time.time())}_seed{int(seed)}.png")
#     return result

In [9]:
# Step 4: Real-ESRGAN x4plus upscaling and finishing controls
ESRGAN_PATH = "/kaggle/working/RealESRGAN_x4plus.pth"
ESRGAN_URL = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"

class ResidualDenseBlock(nn.Module):
    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.conv1 = nn.Conv2d(num_feat, num_grow_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_feat + num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_feat + 2 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_feat + 3 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_feat + 4 * num_grow_ch, num_feat, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x

class RRDB(nn.Module):
    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb2 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb3 = ResidualDenseBlock(num_feat, num_grow_ch)

    def forward(self, x):
        out = self.rdb3(self.rdb2(self.rdb1(x)))
        return out * 0.2 + x

class RRDBNet(nn.Module):
    def __init__(self, num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32):
        super().__init__()
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)
        self.body = nn.Sequential(*[RRDB(num_feat, num_grow_ch) for _ in range(num_block)])
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        feat = self.conv_first(x)
        feat = feat + self.conv_body(self.body(feat))
        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode="nearest")))
        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode="nearest")))
        return self.conv_last(self.lrelu(self.conv_hr(feat)))

def load_upscaler():
    if not os.path.exists(ESRGAN_PATH):
        urllib.request.urlretrieve(ESRGAN_URL, ESRGAN_PATH)
    net = RRDBNet()
    checkpoint = torch.load(ESRGAN_PATH, map_location="cpu", weights_only=True)
    state_dict = checkpoint.get("params_ema", checkpoint.get("params", checkpoint))
    net.load_state_dict(state_dict, strict=True)
    return net.eval().to("cuda")

@torch.no_grad()
def upscale_tiled(net, img, tile=256, pad=32):
    """Upscale one PIL image 4x in tiles; works for portrait, landscape, and odd dimensions."""
    scale = 4
    if tile <= 0 or pad < 0:
        raise ValueError("tile must be positive and pad cannot be negative")

    rgb = img.convert("RGB")
    arr = np.asarray(rgb, dtype=np.float32).copy() / 255.0
    input_tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    first_param = next(net.parameters(), None)
    device = first_param.device if first_param is not None else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    input_tensor = input_tensor.to(device)

    input_height, input_width = input_tensor.shape[-2:]
    output = torch.empty((1, 3, input_height * scale, input_width * scale), dtype=torch.float32, device="cpu")

    for top in range(0, input_height, tile):
        bottom = min(top + tile, input_height)
        tile_top = max(0, top - pad)
        tile_bottom = min(input_height, bottom + pad)
        core_height = (bottom - top) * scale
        crop_top = (top - tile_top) * scale

        for left in range(0, input_width, tile):
            right = min(left + tile, input_width)
            tile_left = max(0, left - pad)
            tile_right = min(input_width, right + pad)
            core_width = (right - left) * scale
            crop_left = (left - tile_left) * scale

            input_tile = input_tensor[:, :, tile_top:tile_bottom, tile_left:tile_right]
            output_tile = net(input_tile).to(device="cpu", dtype=torch.float32)
            expected_tile_shape = ((tile_bottom - tile_top) * scale, (tile_right - tile_left) * scale)
            if output_tile.ndim != 4 or output_tile.shape[0] != 1 or output_tile.shape[1] != 3 or output_tile.shape[-2:] != expected_tile_shape:
                raise RuntimeError(
                    f"Upscaler returned tile shape {tuple(output_tile.shape)}; expected 1x3x{expected_tile_shape[0]}x{expected_tile_shape[1]}."
                )

            core = output_tile[:, :, crop_top:crop_top + core_height, crop_left:crop_left + core_width]
            if core.shape[-2:] != (core_height, core_width):
                raise RuntimeError(
                    f"Tile crop is {tuple(core.shape[-2:])}; expected {(core_height, core_width)}. "
                    f"Check tile bounds: image={input_width}x{input_height}, x={left}:{right}, y={top}:{bottom}."
                )
            output[:, :, top * scale:bottom * scale, left * scale:right * scale] = core

    output = output.clamp(0, 1).squeeze(0).permute(1, 2, 0).numpy()
    return Image.fromarray((output * 255).round().astype(np.uint8))

def finish_image(img, sharpen, contrast, warmth, grain):
    if img is None:
        return gr.skip()
    out = ImageEnhance.Contrast(img).enhance(float(contrast))
    out = ImageEnhance.Sharpness(out).enhance(1.0 + float(sharpen))
    arr = np.asarray(out.convert("RGB"), dtype=np.float32).copy()
    warmth = float(warmth)
    arr[:, :, 0] += warmth
    arr[:, :, 2] -= warmth
    grain = float(grain)
    if grain > 0:
        grain_seed = zlib.adler32(img.tobytes())
        noise = np.random.default_rng(grain_seed).normal(0, grain, size=(arr.shape[0], arr.shape[1], 1))
        arr += noise.astype(np.float32)
    return Image.fromarray(np.clip(arr, 0, 255).round().astype(np.uint8))

def upscale_image(img, factor, sharpen, contrast, warmth, grain):
    if img is None:
        raise gr.Error("Generate an image first.")
    net = load_upscaler()
    try:
        base = upscale_tiled(net, img)
    finally:
        del net
        free_gpu()
    if factor == "2x":
        base = base.resize((base.width // 2, base.height // 2), Image.Resampling.LANCZOS)
    finished = finish_image(base, sharpen, contrast, warmth, grain)
    return finished, base


In [ ]:
css = ".gradio-container {max-width: 680px !important; margin: 0 auto !important;}"

with gr.Blocks(title="Editorial campaign generator") as demo:
    gr.Markdown("### Prompt Generation")
    with gr.Row():
        product = gr.Image(type="filepath", label="Product photo reference", height=260)
        moods = gr.File(file_count="multiple", type="filepath", label="Mood board", height=260)
    request = gr.Textbox(label="Request for Qwen (optional)", value="Editorial realistic campaign shot, model wearing this product.")
    framing = gr.Dropdown(["", "full-body", "three-quarter length", "waist-up", "close-up detail", "close-up face"], value="full-body", label="Framing")
    with gr.Row():
        lighting = gr.Textbox(label="Lighting (optional)", placeholder="hard direct flash")
        camera = gr.Textbox(label="Camera (optional)", placeholder="35mm film, low angle")
    with gr.Row():
        setting = gr.Textbox(label="Setting (optional)", placeholder="studio gray background")
        palette = gr.Textbox(label="Palette (optional)", placeholder="greens")
    btn_caption = gr.Button("GENERATE PROMPT WITH QWEN", variant="secondary")

    gr.Markdown("### Image Generation")
    caption_box = gr.Textbox(label="Prompt", placeholder="Write your image prompt here, or use the optional Qwen button above.", lines=10)
    with gr.Row():
        aspect = gr.Dropdown(["1:1", "2:3", "3:4", "4:3", "4:5", "16:9"], value="1:1", label="Aspect ratio")
        seed = gr.Slider(1, 9999, value=1, step=1, label="Seed")
    use_mood_ref = gr.Checkbox(label="Pass up to moodboard", value=False)
    btn_gen = gr.Button("GENERATE IMAGE", variant="primary")
    result = gr.Image(label="Result", type="pil", height=400)
    current_img = gr.State(None)

    # gr.Markdown("### Edit Image")
    # btn_open_editor = gr.Button("EDIT IMAGE", variant="secondary")
    # with gr.Column(visible=False) as edit_panel:
    #     editor = gr.ImageEditor(type="pil", label="Paint over the area to change", width=640, height=700,
    #                             canvas_size=(640, 640), fixed_canvas=False,
    #                             brush=gr.Brush(colors=[("#ff0000", 0.45)], default_color=("#ff0000", 0.45), default_size=25, color_mode="fixed"),
    #                             sources=[], transforms=[])
    #     instruction = gr.Textbox(label="What should change in the painted area?", placeholder="replace the bag with a small red leather clutch", lines=2)
    #     btn_edit_caption = gr.Button("UPDATE PROMPT", variant="primary")
    #     edit_caption_box = gr.Textbox(label="New prompt (editable)", lines=5)
    #     with gr.Row():
    #         edit_seed = gr.Slider(1, 9999, value=1, step=1, label="Seed")
    #         feather = gr.Slider(0, 30, value=8, step=1, label="Edge softness (px)")
    #     with gr.Row():
    #         add_product = gr.Checkbox(label="Also use the product photo as a reference", value=False)
    #     with gr.Row():
    #         btn_edit = gr.Button("APPLY EDIT", variant="primary")
    #         btn_close_editor = gr.Button("CLOSE EDITOR")

    gr.Markdown("### FINISH AND UPSCALE")
    with gr.Row():
        upscale_factor = gr.Radio(["2x", "4x"], value="2x", label="Upscale factor")
    btn_up = gr.Button("UPSCALE IMAGE", variant="primary")
    upscaled = gr.Image(label="Finished image", type="pil", height=400)
    upscaled_base = gr.State(None)
    with gr.Row():
        finish_contrast = gr.Slider(0.85, 1.20, value=1.03, step=0.01, label="Contrast")
        finish_sharpen = gr.Slider(0.0, 0.8, value=0.15, step=0.05, label="Sharpen")
    with gr.Row():
        finish_warmth = gr.Slider(-15, 15, value=0, step=1, label="Color warmth")
        finish_grain = gr.Slider(0, 12, value=2, step=0.5, label="Film grain")

    btn_caption.click(write_caption, [product, moods, request, framing, lighting, camera, setting, palette], caption_box)
    btn_gen.click(generate_and_store, [product, moods, caption_box, use_mood_ref, seed, aspect], [result, current_img]).then(lambda: (None, None), outputs=[upscaled_base, upscaled])

    #btn_open_editor.click(reveal_edit_panel, current_img, [edit_panel, editor])
    #btn_close_editor.click(lambda: gr.update(visible=False), outputs=edit_panel)
    #btn_edit_caption.click(edit_caption, [caption_box, instruction, current_img], edit_caption_box)
    #btn_edit.click(edit_and_store, [current_img, editor, edit_caption_box, add_product, product, edit_seed, feather], [result, current_img]).then(lambda t: t, edit_caption_box, caption_box).then(image_to_editor_value, current_img, editor).then(lambda: (None, None), outputs=[upscaled_base, upscaled])

    btn_up.click(upscale_image, [current_img, upscale_factor, finish_sharpen, finish_contrast, finish_warmth, finish_grain], [upscaled, upscaled_base])
    for finish_control in [finish_sharpen, finish_contrast, finish_warmth, finish_grain]:
        finish_control.release(finish_image, [upscaled_base, finish_sharpen, finish_contrast, finish_warmth, finish_grain], upscaled)

demo.launch(share=True, debug=True, theme=gr.themes.Soft(), css=css)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8e30f1e01935d23c88.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1963, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^^^^^^^^^

modular_model_index.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0k [00:00<?, ?B/s]

... (20 outputs left out)

  0%|          | 0/16 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
The config attributes {'force_upcast': True} were passed to AutoencoderKLFlux2, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading weights:   0%|          | 0/1425 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]